# Chapter 6: MLOps Pipeline

**From *AI Enterprise Architect***

Build a simple MLOps pipeline: train a model, log experiments, evaluate against a test
set, register the model, and simulate a deployment decision gate. This notebook shows
how CI/CD discipline applies to model deployment.

## What You'll Learn
- How to build a lightweight experiment tracking and model registry system
- Training and evaluating a text classifier (TF-IDF + Logistic Regression)
- Deployment gates: automated go/no-go based on metric thresholds
- Model card generation for documentation and compliance
- Model versioning and comparison

## Prerequisites
- Basic Python and scikit-learn knowledge
- No API keys required for this notebook

In [ ]:
# Install dependencies
!pip install -q scikit-learn numpy

## 1. Setup and Imports

We use scikit-learn for the ML model and build our own lightweight experiment tracker
and model registry. In production you would use MLflow, Weights & Biases, or a similar
platform, but the concepts are identical.

In [ ]:
import os
import json
import numpy as np
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Any, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.pipeline import Pipeline

print("All imports successful.")

## 2. Training and Test Data

We create 50 labeled training examples and 20 test examples for a simple sentiment
classifier (positive/negative). The data simulates enterprise feedback — product
reviews, employee surveys, and customer comments.

In [ ]:
# --- Training data: 50 labeled examples ---
TRAIN_DATA = [
    # Positive examples (25)
    ("The new platform is incredibly efficient and easy to use", "positive"),
    ("Great improvement in system performance after the upgrade", "positive"),
    ("Our team loves the new dashboard features", "positive"),
    ("Excellent customer support experience with fast resolution", "positive"),
    ("The migration went smoothly with minimal downtime", "positive"),
    ("Very satisfied with the API response times", "positive"),
    ("The documentation is clear and well organized", "positive"),
    ("Outstanding reliability and uptime this quarter", "positive"),
    ("The new features exceeded our expectations", "positive"),
    ("Seamless integration with our existing tools", "positive"),
    ("The onboarding process was quick and intuitive", "positive"),
    ("Significant cost savings compared to the previous solution", "positive"),
    ("The AI recommendations are surprisingly accurate", "positive"),
    ("Really impressed with the scalability improvements", "positive"),
    ("The user interface is clean and modern", "positive"),
    ("Our productivity has increased noticeably since adoption", "positive"),
    ("The security features give us confidence in compliance", "positive"),
    ("Fantastic ROI on this technology investment", "positive"),
    ("The automated reports save hours of manual work every week", "positive"),
    ("Smooth rollout with excellent change management support", "positive"),
    ("The platform handles peak loads without any issues", "positive"),
    ("Very happy with the vendor partnership and communication", "positive"),
    ("The data quality has improved dramatically", "positive"),
    ("Best enterprise tool we have deployed in years", "positive"),
    ("The training program helped our team get up to speed fast", "positive"),
    # Negative examples (25)
    ("The system crashes frequently during peak hours", "negative"),
    ("Poor documentation makes it hard to troubleshoot issues", "negative"),
    ("The upgrade caused multiple integration failures", "negative"),
    ("Customer support response times are unacceptably slow", "negative"),
    ("The user interface is confusing and hard to navigate", "negative"),
    ("Frequent timeouts when processing large datasets", "negative"),
    ("The API is unreliable and returns inconsistent results", "negative"),
    ("Migration took twice as long as estimated with many bugs", "negative"),
    ("The cost overruns on this project are concerning", "negative"),
    ("Security vulnerabilities were found after deployment", "negative"),
    ("The performance degraded significantly after the update", "negative"),
    ("Our team struggles with the complexity of this tool", "negative"),
    ("Lack of customization options limits its usefulness", "negative"),
    ("The reporting feature is buggy and often shows wrong data", "negative"),
    ("Vendor communication has been poor throughout the project", "negative"),
    ("The system does not scale well under heavy load", "negative"),
    ("Data loss occurred during the migration process", "negative"),
    ("The onboarding experience was frustrating and confusing", "negative"),
    ("Too many manual steps in what should be an automated workflow", "negative"),
    ("The platform is slow and unresponsive during business hours", "negative"),
    ("Integration with existing systems required extensive workarounds", "negative"),
    ("The promised features were not delivered on schedule", "negative"),
    ("Compliance reporting is incomplete and requires manual fixes", "negative"),
    ("The model predictions are unreliable and often incorrect", "negative"),
    ("Downtime has increased significantly since the last release", "negative"),
]

# --- Test data: 20 labeled examples (held out) ---
TEST_DATA = [
    ("The new analytics feature provides great insights", "positive"),
    ("System stability has improved remarkably", "positive"),
    ("The deployment pipeline works flawlessly", "positive"),
    ("Excellent performance under high concurrency", "positive"),
    ("The monitoring tools are comprehensive and useful", "positive"),
    ("Users report high satisfaction with the new workflow", "positive"),
    ("The cloud migration delivered expected cost benefits", "positive"),
    ("Real-time processing capabilities are impressive", "positive"),
    ("The collaboration features have improved team efficiency", "positive"),
    ("Strong encryption and access controls meet our standards", "positive"),
    ("The system fails silently without any error messages", "negative"),
    ("Performance bottlenecks make the tool unusable at scale", "negative"),
    ("The upgrade broke several critical integrations", "negative"),
    ("Data synchronization issues cause inconsistent reports", "negative"),
    ("The learning curve is too steep for most users", "negative"),
    ("Frequent outages disrupt our daily operations", "negative"),
    ("The backup and recovery process is unreliable", "negative"),
    ("API rate limits are too restrictive for our use case", "negative"),
    ("The vendor has been unresponsive to our escalations", "negative"),
    ("Technical debt from the initial implementation slows progress", "negative"),
]

# Separate features and labels.
X_train = [text for text, _ in TRAIN_DATA]
y_train = [label for _, label in TRAIN_DATA]
X_test = [text for text, _ in TEST_DATA]
y_test = [label for _, label in TEST_DATA]

print(f"Training set: {len(X_train)} examples ({y_train.count('positive')} pos, {y_train.count('negative')} neg)")
print(f"Test set:     {len(X_test)} examples ({y_test.count('positive')} pos, {y_test.count('negative')} neg)")

## 3. SimpleModelRegistry: Lightweight Experiment Tracking

In production you would use MLflow, but setting up a tracking server in Colab adds
friction. Instead, we build a `SimpleModelRegistry` class that captures the same
concepts: experiment logging, model versioning, and metric comparison.

In [ ]:
@dataclass
class ExperimentRun:
    """Records a single training experiment."""
    run_id: str
    model_name: str
    version: int
    hyperparameters: Dict[str, Any]
    metrics: Dict[str, float]
    training_date: str
    training_samples: int
    test_samples: int
    status: str = "trained"  # trained, approved, deployed, rejected
    notes: str = ""


class SimpleModelRegistry:
    """
    A lightweight model registry that tracks experiments, stores metrics,
    and supports model versioning and comparison.

    Mirrors the core functionality of MLflow Model Registry:
    - Log runs with hyperparameters and metrics
    - Compare runs side by side
    - Promote models through stages (trained -> approved -> deployed)
    """

    def __init__(self):
        self.runs: List[ExperimentRun] = []
        self._version_counter = 0

    def log_run(
        self,
        model_name: str,
        hyperparameters: Dict[str, Any],
        metrics: Dict[str, float],
        training_samples: int,
        test_samples: int,
        notes: str = "",
    ) -> ExperimentRun:
        """Log a new experiment run."""
        self._version_counter += 1
        run = ExperimentRun(
            run_id=f"run-{self._version_counter:03d}",
            model_name=model_name,
            version=self._version_counter,
            hyperparameters=hyperparameters,
            metrics=metrics,
            training_date=datetime.utcnow().isoformat(),
            training_samples=training_samples,
            test_samples=test_samples,
            notes=notes,
        )
        self.runs.append(run)
        return run

    def compare_runs(self) -> None:
        """Print a comparison table of all runs."""
        if not self.runs:
            print("No runs logged yet.")
            return

        # Header
        print(f"{'Run ID':<10} {'Ver':>4} {'Accuracy':>9} {'Precision':>10} "
              f"{'Recall':>7} {'F1':>7} {'Status':<10}")
        print("-" * 65)

        for run in self.runs:
            m = run.metrics
            print(
                f"{run.run_id:<10} {run.version:>4} "
                f"{m.get('accuracy', 0):>9.4f} {m.get('precision', 0):>10.4f} "
                f"{m.get('recall', 0):>7.4f} {m.get('f1', 0):>7.4f} "
                f"{run.status:<10}"
            )

    def get_best_run(self, metric: str = "f1") -> Optional[ExperimentRun]:
        """Return the run with the highest value of the given metric."""
        if not self.runs:
            return None
        return max(self.runs, key=lambda r: r.metrics.get(metric, 0))

    def update_status(self, run_id: str, new_status: str) -> None:
        """Update the deployment status of a run."""
        for run in self.runs:
            if run.run_id == run_id:
                run.status = new_status
                return
        raise ValueError(f"Run {run_id} not found.")


# Create the registry.
registry = SimpleModelRegistry()
print("SimpleModelRegistry created.")

## 4. Train and Evaluate a Model

We use TF-IDF vectorization with Logistic Regression — a solid baseline for text
classification. We wrap everything in a training function so we can easily train
multiple versions with different hyperparameters.

In [ ]:
def train_and_evaluate(
    X_train: List[str],
    y_train: List[str],
    X_test: List[str],
    y_test: List[str],
    max_features: int = 500,
    ngram_range: Tuple[int, int] = (1, 1),
    C: float = 1.0,
    max_iter: int = 1000,
) -> Tuple[Pipeline, Dict[str, float]]:
    """
    Train a TF-IDF + LogisticRegression pipeline and evaluate on the test set.

    Args:
        max_features: Maximum number of TF-IDF features.
        ngram_range: Range of n-grams (e.g., (1,2) for unigrams + bigrams).
        C: Regularization strength (inverse). Higher = less regularization.
        max_iter: Maximum iterations for the solver.

    Returns:
        The trained pipeline and a dictionary of evaluation metrics.
    """
    # Build the pipeline.
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)),
        ("classifier", LogisticRegression(C=C, max_iter=max_iter, random_state=42)),
    ])

    # Train.
    pipeline.fit(X_train, y_train)

    # Predict on test set.
    y_pred = pipeline.predict(X_test)

    # Calculate metrics.
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label="positive"),
        "recall": recall_score(y_test, y_pred, pos_label="positive"),
        "f1": f1_score(y_test, y_pred, pos_label="positive"),
    }

    return pipeline, metrics


print("Training function defined.")

## 5. Model Versioning: Train 3 Versions with Different Hyperparameters

We train three model versions, each with different hyperparameters, to demonstrate
how experiment tracking helps you pick the best model.

In [ ]:
# Define three sets of hyperparameters to compare.
EXPERIMENTS = [
    {
        "name": "Baseline (unigrams, default regularization)",
        "params": {"max_features": 200, "ngram_range": (1, 1), "C": 1.0},
    },
    {
        "name": "Bigrams with more features",
        "params": {"max_features": 500, "ngram_range": (1, 2), "C": 1.0},
    },
    {
        "name": "Bigrams + stronger regularization",
        "params": {"max_features": 500, "ngram_range": (1, 2), "C": 0.5},
    },
]

trained_models = {}  # Store the pipeline objects for later use.

for experiment in EXPERIMENTS:
    print(f"\nTraining: {experiment['name']}")
    print(f"  Hyperparameters: {experiment['params']}")

    pipeline, metrics = train_and_evaluate(
        X_train, y_train, X_test, y_test, **experiment["params"]
    )

    # Log the run to the registry.
    # Convert tuple to string for JSON serialization.
    serializable_params = {
        k: str(v) if isinstance(v, tuple) else v
        for k, v in experiment["params"].items()
    }
    run = registry.log_run(
        model_name="sentiment-classifier",
        hyperparameters=serializable_params,
        metrics=metrics,
        training_samples=len(X_train),
        test_samples=len(X_test),
        notes=experiment["name"],
    )

    trained_models[run.run_id] = pipeline

    print(f"  Results: accuracy={metrics['accuracy']:.4f}, "
          f"precision={metrics['precision']:.4f}, "
          f"recall={metrics['recall']:.4f}, f1={metrics['f1']:.4f}")
    print(f"  Logged as {run.run_id} (version {run.version})")

print("\n" + "=" * 65)
print("All experiments complete. Comparison:")
print("=" * 65)
registry.compare_runs()

## 6. Detailed Test Set Evaluation

Let's look at the full classification report for the best model, including
per-class precision, recall, and F1.

In [ ]:
# Get the best model.
best_run = registry.get_best_run(metric="f1")
best_pipeline = trained_models[best_run.run_id]

print(f"Best model: {best_run.run_id} (version {best_run.version})")
print(f"Notes: {best_run.notes}")
print(f"Hyperparameters: {json.dumps(best_run.hyperparameters, indent=2)}")
print()

# Full classification report.
y_pred = best_pipeline.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Show individual predictions.
print("\nPer-sample predictions:")
print(f"{'#':<3} {'Actual':<10} {'Predicted':<10} {'Correct':>8}  Text")
print("-" * 90)
for i, (text, actual) in enumerate(TEST_DATA):
    predicted = y_pred[i]
    correct = "YES" if actual == predicted else "NO"
    print(f"{i+1:<3} {actual:<10} {predicted:<10} {correct:>8}  {text[:55]}")

## 7. Deployment Gate

A deployment gate is an automated check that decides whether a model is ready for
production. The model deploys **only if**:
- Accuracy > 0.80
- F1 score > 0.75

This prevents bad models from reaching users and enforces quality standards.

In [ ]:
def deployment_gate(
    run: ExperimentRun,
    min_accuracy: float = 0.80,
    min_f1: float = 0.75,
) -> Tuple[bool, List[str]]:
    """
    Evaluate whether a model passes the deployment gate.

    Returns:
        (passed, reasons): A boolean and a list of pass/fail reasons.
    """
    checks = []
    all_passed = True

    # Check accuracy.
    acc = run.metrics.get("accuracy", 0)
    if acc >= min_accuracy:
        checks.append(f"PASS: accuracy {acc:.4f} >= {min_accuracy}")
    else:
        checks.append(f"FAIL: accuracy {acc:.4f} < {min_accuracy}")
        all_passed = False

    # Check F1.
    f1 = run.metrics.get("f1", 0)
    if f1 >= min_f1:
        checks.append(f"PASS: f1 {f1:.4f} >= {min_f1}")
    else:
        checks.append(f"FAIL: f1 {f1:.4f} < {min_f1}")
        all_passed = False

    return all_passed, checks


# Run the deployment gate on all models.
print("=== Deployment Gate Results ===")
print()

for run in registry.runs:
    passed, checks = deployment_gate(run)
    decision = "APPROVED for deployment" if passed else "REJECTED"

    # Update the registry status.
    new_status = "approved" if passed else "rejected"
    registry.update_status(run.run_id, new_status)

    print(f"{run.run_id} (v{run.version}): {decision}")
    for check in checks:
        print(f"  {check}")
    print()

# Show updated registry.
print("Updated registry:")
registry.compare_runs()

## 8. Select and Deploy the Best Approved Model

Among all approved models, we pick the one with the highest F1 score and promote
it to "deployed" status.

In [ ]:
# Find the best approved model.
approved_runs = [r for r in registry.runs if r.status == "approved"]

if approved_runs:
    best_approved = max(approved_runs, key=lambda r: r.metrics.get("f1", 0))
    registry.update_status(best_approved.run_id, "deployed")
    print(f"Deployed: {best_approved.run_id} (v{best_approved.version})")
    print(f"  Accuracy: {best_approved.metrics['accuracy']:.4f}")
    print(f"  F1:       {best_approved.metrics['f1']:.4f}")
    print(f"  Notes:    {best_approved.notes}")
else:
    print("No models passed the deployment gate. Review and retrain.")
    best_approved = None

print("\nFinal registry state:")
registry.compare_runs()

## 9. Model Card Generation

A model card is a standardized document that describes a model's intended use,
performance characteristics, limitations, and training details. It is increasingly
required by regulations (EU AI Act) and internal governance policies.

We auto-generate one from the logged metadata.

In [ ]:
def generate_model_card(run: ExperimentRun) -> str:
    """
    Generate a markdown model card from experiment metadata.
    In production, this would be stored in a model registry or
    artifact store alongside the model weights.
    """
    card = f"""# Model Card: {run.model_name}

## Overview
- **Model Name**: {run.model_name}
- **Version**: {run.version}
- **Run ID**: {run.run_id}
- **Status**: {run.status}
- **Training Date**: {run.training_date}

## Intended Use
Binary sentiment classification of enterprise feedback text (positive/negative).
Designed for internal use in customer feedback analysis and employee survey processing.

## Training Data
- **Training samples**: {run.training_samples}
- **Test samples**: {run.test_samples}
- **Data source**: Curated enterprise feedback corpus
- **Label distribution**: Balanced (50/50 positive/negative)

## Hyperparameters
"""
    for key, value in run.hyperparameters.items():
        card += f"- **{key}**: {value}\n"

    card += f"""
## Performance Metrics
| Metric    | Value  |
|-----------|--------|
| Accuracy  | {run.metrics['accuracy']:.4f} |
| Precision | {run.metrics['precision']:.4f} |
| Recall    | {run.metrics['recall']:.4f} |
| F1 Score  | {run.metrics['f1']:.4f} |

## Limitations
- Trained on English text only
- Binary classification only (no neutral category)
- Small training set ({run.training_samples} examples) — may not generalize to all domains
- TF-IDF features do not capture word order or semantic meaning

## Ethical Considerations
- Model should not be used for high-stakes decisions without human review
- Sentiment labels may reflect cultural bias in training data
- Regular retraining recommended as language patterns evolve

## Deployment Gate
- Required accuracy: > 0.80
- Required F1: > 0.75
- Gate result: **{run.status.upper()}**

---
*Generated automatically by SimpleModelRegistry on {datetime.utcnow().isoformat()}*
"""
    return card


# Generate the model card for the deployed model (or best model if none deployed).
card_run = best_approved if best_approved else registry.get_best_run("f1")
model_card = generate_model_card(card_run)

print(model_card)

## 10. Full Experiment Log (JSON Export)

In a real MLOps system, all experiment data is stored in a database or artifact
store. Here we export it as JSON to show the full data model.

In [ ]:
# Export all experiment data as JSON.
experiment_log = {
    "project": "sentiment-classifier",
    "total_runs": len(registry.runs),
    "export_date": datetime.utcnow().isoformat(),
    "deployment_gate": {
        "min_accuracy": 0.80,
        "min_f1": 0.75,
    },
    "runs": [asdict(run) for run in registry.runs],
}

print(json.dumps(experiment_log, indent=2))

## Key Takeaways

1. **MLOps brings CI/CD discipline to model deployment.** Just as you wouldn't deploy
   code without tests, you shouldn't deploy a model without automated quality gates.

2. **Experiment tracking is essential.** Without it, you lose track of which
   hyperparameters produced which results, leading to wasted compute and irreproducible
   experiments.

3. **Deployment gates prevent bad models from reaching production.** Define clear
   thresholds and automate the go/no-go decision.

4. **Model cards document what a model does and doesn't do.** They are increasingly
   required by regulation and are good engineering practice regardless.

5. **Model versioning enables rollback.** When a new model underperforms in production,
   you can quickly revert to the previous version — just like code deployments.

### For Enterprise Architects

MLOps is not just a data science concern. As an AI Enterprise Architect, you design
the platform that teams use to train, evaluate, register, and deploy models. Your
architecture should include: a model registry, an experiment tracker, automated
deployment gates, model monitoring, and a rollback mechanism. These are the same
patterns you use for application CI/CD — adapted for ML artifacts.